# Section 4: Toxic comment classification: Bi-LSTM with GloVe

Trains a bidirectional LSTM on the pooled toxicity data from `3_pooled_data.ipynb` (YouTube, Wikipedia, news and Dota chat comments with a binary `label`), tunes it with Bayesian search and k-fold cross-validation, and evaluates the best model on the pooled validation split and zero-shot on r/jobs comments.

Pipeline: load the pooled train / validation splits → tokenize (fitted on the training text) and pad → GloVe embedding matrix → Bayesian search (each candidate scored by k-fold CV on the training split) → evaluate the best model on the validation split → Reddit zero-shot test.

Run `3_pooled_data.ipynb` first: it creates `dataset/pooled/pooled_train.csv`, `pooled_val.csv` and `manifest.json`.

## 1. Setup
Imports, the shared project code in the `functions/` package (`preprocessing`, `embeddings`, `cv`, `tuning`), and a filter for a harmless Keras warning about masking.

In [ ]:
# Add the parent directory ('Project Template') to Python's search path
import sys
sys.path.append('..')

# project code: see functions/__init__.py for what each module contains
from functions.preprocessing import cleanup_text
from functions.embeddings import build_embedding_matrix, download_glove, glove_filename
from functions.cv import kfold_wrapper
from functions.tuning import bayes_search


import os
import re
import json
import hashlib
import string
import warnings
import numpy as np
import unicodedata
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Embedding
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from skopt.space import Categorical, Integer, Real
import matplotlib.pyplot as plt

# the nightly Keras build wrongly warns that LSTM / Bidirectional destroy the mask (masking works)
warnings.filterwarnings('ignore', message='.*was passed an input with a mask attached.*')

## 2. Load the pooled data
Reads the fixed train / validation splits written by `3_pooled_data.ipynb` to `../dataset/pooled/`. Each row has `orig_id`, `source`, `text_raw`, `text` (lower-cased and cleaned) and `label` (1 = toxic). The splits are stratified by source and label, so training and validation have the same toxic share within every source. The files are checked against `manifest.json`, so the notebook stops if they were edited or regenerated differently. The table shows the number of rows and the toxic share per source and split.

In [ ]:
pooled_dir = '../dataset/pooled'
train_path = os.path.join(pooled_dir, 'pooled_train.csv')
val_path = os.path.join(pooled_dir, 'pooled_val.csv')
manifest_path = os.path.join(pooled_dir, 'manifest.json')
for path in (train_path, val_path, manifest_path):
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path} not found. Run notebook/3_pooled_data.ipynb first to create the pooled splits.')

def read_pooled(path):
    # orig_id must stay a string and empty strings must not become NaN
    return pd.read_csv(path, dtype={'orig_id': str}, keep_default_na=False)

train_df, val_df = read_pooled(train_path), read_pooled(val_path)

# the splits must be the ones recorded in the manifest (data, mapping or cleaning may have changed since)
with open(manifest_path, encoding='utf-8') as f:
    manifest = json.load(f)
for name, frame in [('train', train_df), ('val', val_df)]:
    digest = hashlib.sha256(frame.to_csv(index=False, lineterminator='\n').encode()).hexdigest()
    if digest != manifest['split_sha256'][name]:
        raise ValueError(f'pooled_{name}.csv does not match manifest.json: regenerate the splits with notebook 3.')

summary = (pd.concat([train_df.assign(split='train'), val_df.assign(split='val')])
           .groupby(['source', 'split'])['label'].agg(rows='size', toxic_share='mean').unstack('split'))
summary.round(3)

## 3. Check the cleaned text
The pooled `text` column was already produced by `cleanup_text` in notebook 3 (lower-case, HTML and URL removal, contraction expansion, punctuation handling), so nothing is cleaned again here. A spot check re-cleans a sample of the raw text and stops if the result differs from the stored text, which happens when `cleanup_text` has changed since the splits were made. Prediction applies the same function to new comments, so the training and prediction inputs have to match.

In [ ]:
commentsCol = 'text'   # already lower-cased and cleaned in notebook 3
labelCol = 'label'     # 1 = toxic, 0 = non-toxic

# spot check: the stored text must equal what the current cleanup_text gives for the raw text
check = train_df.sample(min(1000, len(train_df)), random_state=0)
stale = (check['text_raw'].str.lower().apply(cleanup_text) != check[commentsCol]).sum()
if stale:
    raise ValueError(f'{stale} of {len(check)} sampled texts differ from the current cleanup_text: re-run notebook 3.')
print(f'Cleaning check passed on {len(check)} sampled training comments.')

## 4. Explore the data
Rows and toxic share per source, and the distribution of comment lengths (in words) per source, for the training split. The sources differ a lot: Dota chat messages are only a few words long, while news and Wikipedia comments are much longer (notebook 3 has the exact figures). This informs the padding length (`max_length`) chosen in the next step.

In [ ]:
counts = train_df.groupby('source')[labelCol].agg(rows='size', toxic='sum')
lengths = train_df[commentsCol].str.split().str.len()

fig, ax = plt.subplots(figsize=(12, 5), nrows=1, ncols=2)

# rows per source, split into non-toxic and toxic
ax[0].bar(counts.index, counts['rows'] - counts['toxic'], color='#4C72B0', label='non-toxic')
ax[0].bar(counts.index, counts['toxic'], bottom=counts['rows'] - counts['toxic'], color='#DD8452', label='toxic')
for i, (rows, toxic) in enumerate(zip(counts['rows'], counts['toxic'])):
    ax[0].text(i, rows + 50, f'{rows}\n({toxic / rows:.0%} toxic)', ha='center', va='bottom', fontsize=10)
ax[0].set_ylabel('Count')
ax[0].set_title('Training rows per source')
ax[0].legend()

# comment length in words per source (outliers hidden)
ax[1].boxplot([lengths[train_df['source'] == s] for s in counts.index], tick_labels=counts.index, showfliers=False)
ax[1].set_ylabel('Comment length (words)')
ax[1].set_title('Comment length per source')
plt.tight_layout()
plt.show()

## 5. Tokenize and pad
Sets the vocabulary size, the padded length and the embedding dimension, which are reused by the model and the inference function. The tokenizer maps each word to an integer id and is **fitted on the training texts only**, so the validation comments don't influence the vocabulary (the pooled splits exist before this step). Sequences are padded or truncated at the end to a fixed `max_length`, with 0 reserved for padding. The printed shares show how many training comments per source are longer than `max_length` and are therefore truncated.

`embed_dim` must match the GloVe file loaded next.

In [ ]:
# Initialize the tokenizer
vocab_size = 10000
max_length = 80
embed_dim = 100  # must match the GloVe file (50, 100, 200 or 300)

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')

# Fit the tokenizer on the training texts only (this learns the vocabulary)
tokenizer.fit_on_texts(train_df[commentsCol])

# Convert texts to sequences of integers
train_sequences = tokenizer.texts_to_sequences(train_df[commentsCol])
val_sequences = tokenizer.texts_to_sequences(val_df[commentsCol])

# Example of what this looks like
print('Original:', train_df[commentsCol].iloc[0])
print('\nTokenized sequence:', train_sequences[0])

# Pad sequences
X_train = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
X_val = pad_sequences(val_sequences, maxlen=max_length, padding='post', truncating='post')
y_train = train_df[labelCol]
y_val = val_df[labelCol]

print('\nPadded shapes:', X_train.shape, X_val.shape)

# share of training comments cut off at max_length, per source
truncated = train_df[commentsCol].str.split().str.len().gt(max_length)
print('\nTruncated at max_length:')
print(truncated.groupby(train_df['source']).mean().round(3).to_string())

## 6. Load GloVe embeddings
Builds the embedding matrix that initialises the model's embedding layer: row *i* holds the pre-trained GloVe vector of the word with id *i*. It uses the local copy in `../glove` and downloads the file there first if it is missing. Words GloVe doesn't know get small random vectors. The printed coverage is the share of distinct vocabulary words found in GloVe.

In [ ]:
# Load pre-trained GloVe vectors, preferring the local copy in ../glove
glove_dir = '../glove'
glove_path = os.path.join(glove_dir, glove_filename(embed_dim))

try:
    embedding_matrix, glove_coverage = build_embedding_matrix(glove_path, tokenizer.word_index, vocab_size)
except FileNotFoundError:
    print(f'{glove_path} not found.')
    download_glove(embed_dim, glove_dir)
    embedding_matrix, glove_coverage = build_embedding_matrix(glove_path, tokenizer.word_index, vocab_size)

print('Embedding matrix shape:', embedding_matrix.shape)
print(f'Vocabulary found in GloVe: {glove_coverage:.1%}')

## 7. Reference model
A fixed example of the architecture: GloVe-initialised embedding (frozen, with padding masked) → Bi-LSTM → sigmoid output, compiled with several metrics. The search below rebuilds the model itself through `build_model`, so this cell is not needed for the search or for evaluating the best model. It shows the architecture and its summary.

In [ ]:
# 1. Define parameters based on tokenization step
v_size = (
    vocab_size  # Max number of unique words your tokenizer tracked (e.g., top 5,000)
)
embedding_dim = (
    embed_dim  # The number of dimensions (coordinates) for each word vector
)
m_length = (
    max_length  # The fixed length of your padded sequences (e.g., 50 words per comment)
)

# 2. Build the Model Architecture
model = Sequential([
    # The Embedding Layer: integer-to-vector lookup, initialised with GloVe and frozen;
    # mask_zero makes the LSTM skip the padding (index 0)
    Embedding(
        input_dim=v_size,
        output_dim=embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        trainable=False,
        mask_zero=True,
    ),
    # The Bi-LSTM Layer: reads the sequence forwards and backwards, then concatenates both outputs
    Bidirectional(LSTM(8, dropout=0.2, recurrent_dropout=0.2)),
    # The Output Layer: Binary classification (toxic = 1, not toxic = 0)
    Dense(1, activation='sigmoid'),
])

# 3. Compile the model
model.compile(
loss='binary_crossentropy',
    optimizer='adam',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),  # Recall = True Positive Rate
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.F1Score(name='f1_score', threshold=0.5)
    ]
)

# Print the model summary to see the shapes match up
model.summary()

## 8. Train / validation data
The training and validation sets come from notebook 3 (80/20, stratified by source and label), so nothing is split here. The validation split is never used for training or tuning. K-fold cross-validation runs inside the training split only, and its folds are stratified by label, not by source. The commented-out block is a quick manual test of `kfold_wrapper` on the reference model.

In [ ]:
# the pooled splits come from notebook 3 (stratified by source and label); nothing is split here
print(f'Training: {X_train.shape[0]:,} comments | validation: {X_val.shape[0]:,} comments')
print(f'Toxic share: training {y_train.mean():.3f} | validation {y_val.mean():.3f}')

# test the k-fold cross-validation wrapper function
'''
print("Evaluating model...")
evaluated_object = kfold_wrapper(model, X_train, y_train, X_val, y_val, n_splits = 5, epochs = 5, batch_size = 16)
print(evaluated_object.avg_train_metrics)
print(evaluated_object.validation_metrics)
'''

## 9. Hyperparameter search
Defines how a model is built from a set of hyperparameters (`build_model`) and the search space (embedding trainable or frozen, LSTM size, dropout, learning rate). The Bayesian search scores each candidate with stratified k-fold CV on the pooled training split and maximises the average validation PR-AUC.

**Choosing the model.** The top trials are usually within noise of each other, so the search selects, among the trials whose mean PR-AUC is within one fold standard deviation of the best, the one with the lowest standard deviation across folds (`selection='stable'`). It returns that trial's `kfold_wrapper`, whose final model is trained on the full training split and scored on the pooled validation split.

**Choosing the threshold.** The model's cutoff is chosen on the out-of-fold predictions of the training split: the highest-recall cutoff whose false positive rate (the share of non-toxic comments flagged) stays at or below `MAX_FPR`. The cap reflects how many false alarms moderators can review. A recall-weighted score such as F2 was tried first on the YouTube data, but it ran to the lowest cutoff and flagged most comments. The tuned value becomes the model's `threshold`, so the confusion matrix, precision–recall marker, error table, `toxicity_check` and the Reddit section below all use it. The tuning needs no re-training, so it also runs on a loaded result and you can change `MAX_FPR` without re-running the search. The hold-out numbers at the default 0.5 cutoff are printed too, for comparison.

**What the validation scores mean.** The pooled validation split contains comments from the same four sources as training, so its scores are in-domain and optimistic about r/jobs. Section 14 is the cross-domain test.

**Runtime.** The training split has about 19,000 comments instead of the 800 of the earlier YouTube-only setup, so every fit takes much longer, and the search fits about 180 models. To use the CPU better, `kfold_kwargs` has `'n_jobs': -1`: the 5 folds and the final model of each candidate are trained in separate processes at the same time. Each process needs its own memory, so use a smaller number if memory is short, or `1` to train one model after another. The results do not depend on `n_jobs`.

Results are saved to `../models/bayes_search_pooled`. If a saved result exists there, you are asked whether to load it or re-run the search. Choose to re-run after changing the data, preprocessing or search space, because a loaded result is not checked against them. The YouTube-only result in `../models/bayes_search` is not used.

In [ ]:
def build_model(trainable, lstm_units, dropout, lr):
    m = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
            embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
            trainable=trainable,
            mask_zero=True,  # LSTM skips the padding (index 0)
        ),
        Bidirectional(LSTM(lstm_units, dropout=dropout)),
        Dense(1, activation='sigmoid'),
    ])
    m.compile(
        loss='binary_crossentropy',
        optimizer=tf.keras.optimizers.Adam(lr),
        metrics=[
            'accuracy',
            tf.keras.metrics.F1Score(name='f1_score', threshold=0.5),
            tf.keras.metrics.AUC(curve='PR', name='pr_auc'),
        ],
    )
    return m

search_space = {
    'trainable': Categorical([True, False]),  # fine-tune GloVe or keep it frozen
    'lstm_units': Integer(8, 128, prior='log-uniform'),
    'dropout': Real(0.0, 0.5),
    'lr': Real(3e-4, 1e-2, prior='log-uniform'),
    # 'epochs': Integer(10, 60),  # enable after the 5-epoch baseline, and drop epochs from kfold_kwargs
}

# Decision threshold: the highest-recall cutoff whose false positive rate (share of non-toxic
# comments flagged) stays at or below this cap, measured on out-of-fold predictions.
# Tuning needs no re-training, so changing the cap does not require re-running the search.
MAX_FPR = 0.35

# a previous run is saved here; the user chooses whether to load it or re-run the search
# (separate from the YouTube-only search in ../models/bayes_search, which used different data)
save_dir = '../models/bayes_search_pooled'
rerun = True
if os.path.exists(os.path.join(save_dir, 'search_state.pkl')):
    answer = input('Found a previously optimised model. Load it [Y] or re-run the optimisation [N]? (default: Y) ').strip().lower()
    rerun = answer.lower().startswith('n')

if rerun:
    search = bayes_search(
        build_model, search_space, X_train, y_train, X_val, y_val,
        n_iter=30, n_initial_points=10, objective='pr_auc',
        # n_jobs=-1 trains the 5 folds and the final model in separate processes at the same time;
        # each process needs its own memory (watch it), so use a smaller number if memory is short,
        # or 1 to train one model after another
        kfold_kwargs={'n_splits': 5, 'epochs': 5, 'batch_size': 16, 'n_jobs': -1},
        selection='stable',  # among near-ties on mean PR-AUC, take the lowest std across folds
    )
    best_kf = search.fit()
else:
    search = bayes_search.load(save_dir)
    best_kf = search.best_kfold
    print(f'Loaded from {save_dir}')

# the validation set is not used to choose the threshold
if getattr(best_kf, 'oof_proba', None) is not None:
    tuning = best_kf.tune_threshold(max_fpr=MAX_FPR)
else:
    tuning = None
    print('No out-of-fold predictions in this saved result: re-run the search to tune the threshold.')

if rerun:
    search.save(save_dir)
    print(f'Saved to {save_dir}')

print('Selected trial:', getattr(search, 'best_trial', None), '| params:', search.best_params)
print('Hold-out @ threshold 0.5:  ', {k: round(v, 4) for k, v in best_kf.validation_metrics.items()})
if tuning is not None:
    print(f"Tuned threshold: {tuning['threshold']:.3f} ({tuning['rule']}); "
          f"out-of-fold recall {tuning['oof_recall']:.2f}, precision {tuning['oof_precision']:.2f}, FPR {tuning['oof_fpr']:.2f}")
    print('Hold-out @ tuned threshold:', {k: round(v, 4) for k, v in best_kf.tuned_validation_metrics.items()})
search.results.sort_values('avg_val_pr_auc', ascending=False).head()

## 10. Search progress
Validation PR-AUC (mean ± standard deviation across the CV folds) for every round of the search. The dotted line marks the end of the random initial points. Use it to see whether the search is still improving and how large the fold-to-fold noise is relative to the differences between rounds.

In [ ]:
# mean +/- std (across the CV folds) of validation PR-AUC for each round of the search
res = search.results
mean, std = res['avg_val_pr_auc'], res['std_val_pr_auc']

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(res.index, mean, yerr=std, fmt='o-', color='#4C72B0', ecolor='#9db4d6',
            capsize=3, label='mean ± std across folds')
ax.axvline(search.n_initial_points + 0.5, color='grey', linestyle=':', label='end of random initial points')
selected_trial = getattr(search, 'best_trial', None) or mean.idxmax()
ax.scatter([selected_trial], [mean[selected_trial]], color='#DD8452', zorder=3, s=80, label=f'selected (trial {selected_trial})')
ax.set_xlabel('Search round')
ax.set_ylabel('Validation PR-AUC')
ax.set_title('PR-AUC per round of Bayesian search')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Evaluate the best model: confusion matrix
Predicts the held-out validation set with the best model and shows the confusion matrix at the model's threshold. It shows how many toxic comments are caught and how many clean ones are flagged.

In [ ]:
# confusion matrix of the best model on the validation set
from sklearn.metrics import ConfusionMatrixDisplay

proba = best_kf.final_model.predict(X_val, verbose=0).ravel()
pred = (proba >= best_kf.threshold).astype(int)

ConfusionMatrixDisplay.from_predictions(
    np.asarray(y_val).astype(int),
    pred,
    display_labels=['Non-toxic', 'Toxic'],
    cmap='Blues',
)
plt.title('Best model: validation confusion matrix')
plt.show()

### Precision–recall curve and PR-AUC
Plots precision (share of flagged comments that are toxic) against recall (share of toxic comments caught) as the cutoff moves from 1 to 0. PR-AUC (average precision) summarises how well the model ranks toxic above clean comments, independent of any threshold, and it is the metric the search maximises. The dashed line is the toxic share of the validation set, which is the PR-AUC of a model that ranks at random; 1.0 is perfect. The orange dot marks where the model's current threshold sits on the curve, so moving the threshold slides that point along the curve.

In [ ]:
# Precision-recall curve of the best model on the validation set
from sklearn.metrics import precision_recall_curve, average_precision_score

proba = best_kf.final_model.predict(X_val, verbose=0).ravel()
actual = np.asarray(y_val).astype(int)

precision, recall, _ = precision_recall_curve(actual, proba)
pr_auc = average_precision_score(actual, proba)

# operating point at the model's threshold
pred = (proba >= best_kf.threshold).astype(int)
op_recall = ((pred == 1) & (actual == 1)).sum() / (actual == 1).sum()
op_precision = ((pred == 1) & (actual == 1)).sum() / max((pred == 1).sum(), 1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.step(recall, precision, where='post', color='#4C72B0', linewidth=2, label=f'Bi-LSTM (PR-AUC = {pr_auc:.3f})')
ax.axhline(actual.mean(), color='grey', linestyle='--', label=f'Chance (PR-AUC = {actual.mean():.3f})')
ax.scatter([op_recall], [op_precision], color='#DD8452', zorder=3, s=70, label=f'Threshold = {best_kf.threshold:g}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.05)
ax.set_title('Best model: validation precision-recall curve')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

### Cutoff trade-off (out-of-fold)
Shows what each cutoff does to recall, precision, false positive rate and the share of comments flagged, on the out-of-fold predictions of the training split. The row marked `tuned` is the cutoff chosen by the false positive cap. Lowering the cutoff catches more toxic comments but flags more clean ones, so use this table to judge whether the extra recall is worth the extra moderation work. The validation columns are a sanity check only. Training and validation have the same toxic share and the same four sources, so precision here is in-domain; on r/jobs, where the share of toxic comments is unknown and may be lower, precision would differ.

In [ ]:
# Trade-off between catching toxic comments and false alarms at different cutoffs.
# oof_* : out-of-fold predictions on the training set, the basis for choosing the cutoff
# val_* : validation set, for checking only (it did not influence the cutoff)
# recall = share of toxic comments caught | precision = share of flagged comments that are toxic
# fpr = share of non-toxic comments flagged | flagged = share of all comments flagged
best_kf.threshold_table().round(2)

## 12. Error analysis
Lists the mis-classified validation comments with the source, true label, predicted label, predicted probability, cleaned text and length in tokens, sorted by probability. Use it to look for patterns in the errors, such as differences between sources, truncated long comments, sarcasm or implicit toxicity, and noisy labels.

In [ ]:
# mis-classified validation comments of the best model
proba = best_kf.final_model.predict(X_val, verbose=0).ravel()
pred = (proba >= best_kf.threshold).astype(int)
actual = np.asarray(y_val).astype(int)

errors = pd.DataFrame({
    'source': val_df['source'].values,
    'actual_label': actual,
    'predicted_label': pred,
    'predicted_probability': proba,
    'comment': val_df[commentsCol].values,  # cleaned text, as the model saw it
    'n_tokens': val_df[commentsCol].str.split().str.len().values,
}, index=val_df.index)

errors = errors[errors['actual_label'] != errors['predicted_label']]
errors['error_type'] = np.where(errors['actual_label'] == 1, 'missed toxic (FN)', 'false alarm (FP)')

print(errors['error_type'].value_counts().to_string())
print()
print(errors.groupby(['source', 'error_type']).size().unstack(fill_value=0).to_string())
pd.set_option('display.max_colwidth', 200)
errors.sort_values('predicted_probability')

## 13. Predict on new comments
`toxicity_check` applies the same pipeline to new text (clean → tokenize → pad → predict) with the best model and returns "Toxic" or "Non-toxic" using the model's threshold, or one you pass in. It depends on the tokenizer, `max_length` and `cleanup_text` above, so those must match what the model was trained with.

In [ ]:
## pipeline to predict new data
def toxicity_check(new_comments, best_kf, threshold=None):
    # Use the wrapper's cutoff unless another one is given (e.g. a tuned threshold)
    threshold = best_kf.threshold if threshold is None else threshold

    # Clean the new comments
    cleaned_comments = [cleanup_text(comment.lower()) for comment in new_comments]

    # Convert to sequences
    sequences = tokenizer.texts_to_sequences(cleaned_comments)

    # Pad sequences
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

    # Predict with the trained model from the search
    probabilities = best_kf.final_model.predict(padded_sequences, verbose=0).ravel()

    return [{'label': 'Toxic' if p.round(4) >= threshold else 'Non-toxic', 'p': p} for p in probabilities]

### Testing
Runs `toxicity_check` on a sample comment as a quick sanity check.

In [ ]:
toxicity_check(["wouldnt it be nice if you got hit by a bus?",
                "im so sorry to hear that you got hit by a bus",
                "I don’t mean to be insensitive with this question but isn’t it hard to be fired from a government job?",
                "you've got to be a genius to be fired from a government job",
                "you're as smart as a rock"], best_kf)

## 14. Reddit zero-shot validation
Evaluate the selected Bi-LSTM on the shared 100-comment r/jobs sample: **50 non-toxic (`0`) and 50 toxic (`1`) AI preliminary labels**. These are toxicity labels, not sentiment. The current file is pending human review, so its scores measure **agreement with AI annotations**, not accuracy against human-reviewed ground truth.

Run the setup, training-tokenizer and best-model sections above first (or load the matching saved search there). Reuse `best_kf.final_model`, the same `tokenizer`, `max_length`, and the existing `best_kf.threshold`. The threshold was selected before seeing Reddit; this section does not train, fit a tokenizer, tune a threshold or save models.

The sample is balanced and keyword-enriched, with some comments from the same threads. Its precision and PR-AUC do not estimate live r/jobs performance or toxicity prevalence. With only 100 comments, differences of a few points are within noise. Provenance and per-comment label rationales are in `../dataset/reddit/`.

### Load and check the sample
Keep the source text intact. Stop on missing labels, duplicate comments or an unexpected class balance instead of silently treating missing labels as non-toxic.

In [ ]:
from pathlib import Path

reddit_path = Path('../dataset/reddit/test_100_ai_draft.csv')
reddit_df = pd.read_csv(reddit_path, dtype=str, keep_default_na=False)
reddit_required = {'comment_id', 'text', 'subreddit', 'is_toxic', 'permalink',
                   'annotation_status', 'annotator'}
if not reddit_required.issubset(reddit_df.columns):
    raise ValueError(f'Missing Reddit columns: {sorted(reddit_required - set(reddit_df.columns))}')
if len(reddit_df) != 100 or reddit_df['is_toxic'].value_counts().to_dict() != {'0': 50, '1': 50}:
    raise ValueError('Expected exactly 100 labelled comments: 50 non-toxic (0), 50 toxic (1).')
if (reddit_df['comment_id'].str.strip().eq('').any()
        or reddit_df['comment_id'].duplicated().any()
        or reddit_df['text'].str.strip().eq('').any()):
    raise ValueError('Reddit IDs must be nonempty and unique; comment text must be nonempty.')
reddit_text_keys = reddit_df['text'].str.normalize('NFKC').str.casefold().str.split().str.join(' ')
if reddit_text_keys.duplicated().any() or reddit_text_keys.isin(['[deleted]', '[removed]']).any():
    raise ValueError('Duplicate or deleted/removed Reddit comments need review.')
if not reddit_df['subreddit'].eq('jobs').all():
    raise ValueError('This evaluation expects r/jobs comments only.')
if not reddit_df['annotation_status'].isin(['pending', 'reviewed']).all():
    raise ValueError('Exclude rejected comments and resolve invalid review states before evaluation.')

reddit_human_reviewed = (
    reddit_df['annotation_status'].eq('reviewed').all()
    and reddit_df['annotator'].str.strip().ne('').all()
    and not reddit_df['annotator'].str.strip().str.casefold().str.startswith('ai:').any()
)
reddit_label_source = 'Human-reviewed labels' if reddit_human_reviewed else 'AI / unreviewed labels'
print(f'Loaded {len(reddit_df)} r/jobs comments from {reddit_path}')
print(f'Reference: {reddit_label_source}; reviewed status is supplied by the annotation file.')
if not reddit_human_reviewed:
    print('Exploratory agreement only: these labels are not a human-reviewed gold test set.')
display(reddit_df.groupby('is_toxic').size().rename(index={'0': 'Non-toxic', '1': 'Toxic'}).to_frame('count'))

### Apply the training pipeline and predict once
Lowercase → `cleanup_text` → the existing training tokenizer → post-padding/post-truncation to `max_length`. Predict probabilities once and compare the unrounded values with the inherited threshold, just as in the validation-set confusion matrix above. The truncation count shows how much of this sample exceeds the model's input length; it is diagnostic, not a reason to change the trained configuration.

In [ ]:
reddit_dependencies = ['best_kf', 'tokenizer', 'max_length', 'cleanup_text', 'pad_sequences']
reddit_missing = [name for name in reddit_dependencies if name not in globals()]
if reddit_missing:
    raise RuntimeError(f'Run the setup/tokenizer/best-model sections first; missing: {reddit_missing}')
if getattr(best_kf, 'final_model', None) is None or not getattr(tokenizer, 'word_index', None):
    raise RuntimeError('A trained best_kf.final_model and its fitted training tokenizer are required.')
reddit_threshold = float(best_kf.threshold)
if not np.isfinite(reddit_threshold) or not 0 <= reddit_threshold <= 1:
    raise ValueError('The existing model threshold must be finite and between 0 and 1.')

reddit_cleaned = [cleanup_text(text.lower()) for text in reddit_df['text']]
reddit_sequences = tokenizer.texts_to_sequences(reddit_cleaned)
reddit_n_tokens = np.array([len(sequence) for sequence in reddit_sequences])
X_reddit = pad_sequences(reddit_sequences, maxlen=max_length, padding='post', truncating='post')
y_reddit = reddit_df['is_toxic'].astype(int).to_numpy()
reddit_proba = np.asarray(best_kf.final_model.predict(X_reddit, verbose=0), dtype=float).ravel()
if (reddit_proba.shape != y_reddit.shape or not np.isfinite(reddit_proba).all()
        or ((reddit_proba < 0) | (reddit_proba > 1)).any()):
    raise ValueError('The model must return one finite toxicity probability in [0, 1] per comment.')
reddit_pred = (reddit_proba >= reddit_threshold).astype(int)

print(f'Fixed model threshold: {reddit_threshold:g}; input shape: {X_reddit.shape}')
print(f'Comments truncated to {max_length} tokens: {(reddit_n_tokens > max_length).sum()}/{len(reddit_df)}')
print(f'Comments with no tokens after preprocessing: {(reddit_n_tokens == 0).sum()}')

### Metrics, confusion matrix and precision–recall curve
Report the proposal's precision and recall for the **toxic class**, macro F1 over both classes, and **PR-AUC as average precision** (`average_precision_score`), matching `kfold_wrapper`. Use probabilities for PR-AUC and the fixed threshold for the other metrics. The curve describes ranking; do not choose a new threshold from this Reddit sample.

In [ ]:
from sklearn.metrics import (average_precision_score, precision_score, recall_score,
                             f1_score, precision_recall_curve, ConfusionMatrixDisplay)

reddit_metrics = pd.DataFrame([{
    'n_comments': len(y_reddit),
    'threshold': reddit_threshold,
    'precision': precision_score(y_reddit, reddit_pred, zero_division=0),
    'recall': recall_score(y_reddit, reddit_pred, zero_division=0),
    'macro_f1': f1_score(y_reddit, reddit_pred, average='macro', zero_division=0),
    'pr_auc': average_precision_score(y_reddit, reddit_proba),
    'label_source': reddit_label_source,
}], index=['Reddit'])
display(reddit_metrics.round(4))

reddit_precision, reddit_recall, _ = precision_recall_curve(y_reddit, reddit_proba)
fig_reddit, axes_reddit = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_reddit, reddit_pred, labels=[0, 1], display_labels=['Non-toxic', 'Toxic'],
    cmap='Blues', colorbar=False, ax=axes_reddit[0],
)
axes_reddit[0].set_title(f'Reddit confusion matrix (threshold = {reddit_threshold:g})')
axes_reddit[0].set_ylabel('Reference label')
axes_reddit[1].step(reddit_recall, reddit_precision, where='post', color='#4C72B0',
                    label=f"PR-AUC (AP) = {reddit_metrics.loc['Reddit', 'pr_auc']:.3f}")
axes_reddit[1].axhline(y_reddit.mean(), color='grey', linestyle='--', label='Sample prevalence')
axes_reddit[1].scatter(reddit_metrics.loc['Reddit', 'recall'], reddit_metrics.loc['Reddit', 'precision'],
                       color='#DD8452', zorder=3, label='Fixed threshold')
axes_reddit[1].set(xlabel='Recall', ylabel='Precision', title='Reddit precision–recall curve',
                   xlim=(0, 1.02), ylim=(0, 1.05))
axes_reddit[1].legend(loc='lower left')
fig_reddit.suptitle(f'Reference: {reddit_label_source}')
fig_reddit.tight_layout()
plt.show()

### Inspect disagreements with the reference labels
`reddit_predictions` retains all 100 predictions with original text, cleaned input, label status and source links. `reddit_errors` contains only disagreements, with false positives/negatives defined relative to the supplied labels. For this AI-labelled sample, these may be annotation errors as well as model errors. Inspect the sources before drawing conclusions; keep this set out of model/threshold selection.

In [ ]:
reddit_predictions = reddit_df.copy()
reddit_predictions['reference_label'] = y_reddit
reddit_predictions['predicted_label'] = reddit_pred
reddit_predictions['predicted_probability'] = reddit_proba
reddit_predictions['cleaned_text'] = reddit_cleaned
reddit_predictions['n_tokens'] = reddit_n_tokens
reddit_predictions['was_truncated'] = reddit_n_tokens > max_length
reddit_predictions['matches_reference'] = reddit_pred == y_reddit

reddit_errors = reddit_predictions.loc[~reddit_predictions['matches_reference']].copy()
reddit_errors['error_type'] = np.where(reddit_errors['reference_label'] == 1,
                                      'missed toxic (FN)', 'false alarm (FP)')
reddit_errors = reddit_errors.sort_values('predicted_probability')
print(f'Disagreements: {len(reddit_errors)}/{len(reddit_predictions)} against {reddit_label_source}')
print(reddit_errors['error_type'].value_counts().reindex(
    ['missed toxic (FN)', 'false alarm (FP)'], fill_value=0).to_string())
with pd.option_context('display.max_colwidth', 180):
    display(reddit_errors[['comment_id', 'reference_label', 'predicted_label',
                           'predicted_probability', 'error_type', 'n_tokens', 'was_truncated',
                           'text', 'cleaned_text', 'permalink']])